In [1]:
import sys
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
import config
from rx import io, text, places, remote

In [2]:
files_raw = io.read_stage("interim", "files_raw")
pages_index = io.read_stage("interim", "pages_index")
pages_decl = io.read_stage("interim", "pages_decl")
state_mentions = io.read_stage("interim", "state_mentions")

# 2.1
for name, df in {"files_raw": files_raw, "pages_index": pages_index,
                 "pages_decl": pages_decl, "state_mentions": state_mentions}.items():
    print(f"{name:15} {len(df):>9,} rows  {df.NAID.nunique():>6,} files")

# 2.2
print(files_raw.n_pages.describe().round(1))
files_raw.nlargest(3, "n_pages")[["title", "n_pages"]]

# 2.3
src = pages_index[["has_transcription", "has_revx_ocr", "has_fs_text"]]
print((src.mean() * 100).round(1))
print("pages with no text:", (~src.any(axis=1)).sum())
print("files with any transcription: %.1f%%" % (pages_index.groupby("NAID").has_transcription.any().mean() * 100))

# 2.4
print(files_raw.title.str.extract(r"File\s+([A-Z])")[0].value_counts(dropna=False))
files_raw.title[~files_raw.title.str.contains(r"File\s+[A-Z]")].value_counts().head(5)

# 2.5
row = pages_decl.sample(1, random_state=7).iloc[0]
for col in ["transcriptionText", "RevX_OCR_text", "fs_extractedText"]:
    print("----", col)
    print(str(row[col])[:400])


files_raw          78,745 rows  78,745 files
pages_index     2,195,663 rows  78,745 files
pages_decl         62,112 rows  62,112 files
state_mentions    204,846 rows  65,624 files
count    78745.0
mean        27.9
std         23.7
min          1.0
25%         12.0
50%         22.0
75%         39.0
max        609.0
Name: n_pages, dtype: float64
has_transcription    40.3
has_revx_ocr         99.3
has_fs_text          99.5
dtype: float64
pages with no text: 1441
files with any transcription: 40.0%
0
S      36114
W      25622
R      11164
B       3237
NaN     2501
O         61
C         16
N         15
D          4
M          2
P          2
A          2
F          2
J          1
T          1
V          1
Name: count, dtype: int64
---- transcriptionText
nan
---- RevX_OCR_text
State of Maine County of Oxford ss -
On this fourteenth day of September in the year of our Lord eighteen hundred and forty-six, personally appeared before the Honorable Job Prince, Judge of the Court of Probate for sa